# **Notebook 2: Discriptor Calculation**

Mudasir Alvi

In this notebook we have calculate ligands discriptors using:
- RDKIT
- Mordred
- PaDEL

After calculating, merge them into one data set and drop low virance and redudant discriptors.

In [1]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')


# set the path
import os
os.chdir("/content/drive/MyDrive/QSAR_Sigma1R/NoteBooks")

print("Now working in:", os.getcwd())

Mounted at /content/drive
Now working in: /content/drive/MyDrive/QSAR_Sigma1R/NoteBooks


## **RDKIT Descriptors**


In this section, we begin molecular feature extraction using **RDKit**. It is an open-source cheminformatics library originally developed by Greg Landrum and collaborators at Rational Discovery LLC. RDKit has become one of the most widely used toolkits in cheminformatics research and industry because of its speed, open-source nature, and integration with Python and machine learning frameworks.  

RDKit provides a comprehensive suite of functionalities for small-molecule analysis, including:

- **2D molecular descriptors**  
  Numerical features that summarize physicochemical and topological properties of molecules. Examples include molecular weight, logP (hydrophobicity), topological polar surface area (TPSA), number of rotatable bonds, hydrogen bond donors/acceptors, and aromatic ring counts. These descriptors are often directly linked to pharmacokinetics and drug-likeness.  

- **Molecular fingerprints**  
  Binary (0/1) vectors encoding the presence or absence of chemical substructures or motifs. Different fingerprinting methods capture complementary information:  
  - **Morgan Fingerprints (ECFP family)**: Circular fingerprints capturing local atomic environments within a specified radius. Widely used in QSAR modeling and similarity searches.  
  - **MACCS Keys**: A set of 166 predefined substructure patterns (e.g., “hydroxyl group present”).  
  - **Atom Pair Fingerprints**: Encode pairs of atoms and their topological distances, capturing molecular shape and connectivity.  

- **3D conformer generation**  
  RDKit can generate 3D molecular structures from SMILES using the **ETKDG algorithm** (Experimental-Torsion Distance Geometry with knowledge-based rules). These conformers are used for shape-based and 3D descriptor calculations, such as volume, electrostatics, and pharmacophore alignment. Typically, multiple conformers (10–50 per molecule) are generated, energy-minimized, and the lowest-energy structure is retained for downstream analysis.  

📌 **Why combine descriptors and fingerprints?**  
Descriptors provide interpretable, continuous variables (e.g., solubility, lipophilicity, molecular size) that are directly linked to pharmacological behavior. Fingerprints, on the other hand, capture detailed structural motifs that may not be obvious from bulk properties but are critical for receptor binding. By combining both, we ensure a more complete and balanced molecular representation for QSAR modeling.  

📖 **References:**  
- RDKit Documentation: [Getting Started in Python](https://www.rdkit.org/docs/GettingStartedInPython.html)  
- Landrum, G. (2013). *RDKit: Open-source cheminformatics.* http://www.rdkit.org  


In [5]:
# Install rdkit using pip
!pip install rdkit


ERROR: Could not find a version that satisfies the requirement rdkit-pypi (from versions: none)
ERROR: No matching distribution found for rdkit-pypi


In [6]:
# load the data
import pandas as pd

ligands_data = pd.read_csv("../Data/SIR_bioactivity_data_pIC50_03.csv")
ligands_data

,molecule_chembl_id,canonical_smiles,standard_value,standard_value_M,pIC50
0,CHEMBL67010,C/C(=N\C1CCCCC1)NC12CC3CC(CC(C3)C1)C2,72.0,7.200000e-08,7.142668
1,CHEMBL542638,C/C(=N\C12CC3CC(CC(C3)C1)C2)Nc1ccccc1C.Cl,6.0,6.000000e-09,8.221849
2,CHEMBL544054,C/C(=N\C1CCCCC1)Nc1ccccc1C.Cl,9.0,9.000000e-09,8.045757
3,CHEMBL67388,C/C(=N\C12CC3CC(CC(C3)C1)C2)NC12CC3CC(CC(C3)C1)C2,16.0,1.600000e-08,7.795880
4,CHEMBL538754,C/C(=N\c1ccccc1C)Nc1ccccc1C.Cl,15.0,1.500000e-08,7.823909
...,...,...,...,...,...
774,CHEMBL596,CCC(=O)N(c1ccccc1)C1CCN(CCc2ccccc2)CC1,354.0,3.540000e-07,6.450997
775,CHEMBL4761695,[2H]C([2H])([2H])Oc1cc(F)cc2c(N3CCN(c4ccccc4F)...,3500.0,3.500000e-06,5.455932
776,CHEMBL4297224,CN(C)CC1CCOC1(c1ccccc1)c1ccccc1,860.0,8.600000e-07,6.065502
777,CHEMBL4519018,CN1C2CCCC1CC(Nc1ccccc1Br)C2,84.0,8.400000e-08,7.075721


The **RDKit** library provides access to a wide range of descriptors, (more than 200) such as:

- **Physicochemical**: molecular weight, logP, polar surface area  
- **Topological**: connectivity indices, kappa indices, Balaban index  
- **Electronic/structural**: partial charges, atom counts, rotatable bonds  

📌 These descriptors are accessible via RDKit’s built-in functions.  
We can calculate them **individually** (e.g., Lipinski features) or **collectively** (all available descriptors at once). For the list of descritors click the link below.

**References**:  
- RDKit Documentation: [List of available descriptors](https://www.rdkit.org/docs/GettingStartedInPython.html#list-of-available-descriptors)  

## **Calculate Lipinski Descriptors**

In drug discovery, **Lipinski’s Rule of Five** is a well-known guideline proposed by Christopher Lipinski (Pfizer).  
It helps evaluate the **drug-likeness** of compounds based on their pharmacokinetic properties (Absorption, Distribution, Metabolism, Excretion – ADME).

The rule states that an orally active drug is more likely to be successful if it meets the following criteria:

- Molecular weight (MW) < 500 Dalton  
- Octanol-water partition coefficient (LogP) < 5  
- Hydrogen bond donors (HBD) < 5  
- Hydrogen bond acceptors (HBA) < 10  

Compounds that violate more than one of these rules may have poor oral bioavailability.  

📌 In RDKit, these descriptors can be calculated directly using built-in functions.  
We will first compute Lipinski descriptors as a subset of interpretable features before moving to the full descriptor set.


### **Calculate lipinski descriptors**


In [8]:
# function use on the data

ligands_data = pd.read_csv("../Data/SIR_bioactivity_data_pIC50_03.csv")

lipinski_df = calculate_lipinski(ligands_data["canonical_smiles"])
lipinski_df

,MW,LogP,NumHDonors,NumHAcceptors
0,274.452000,4.29590,1,1
1,318.892000,5.21592,1,1
2,266.816000,4.57982,1,1
3,326.528000,4.93200,1,1
4,274.795000,4.88724,1,1
...,...,...,...,...
774,336.479000,4.13670,0,2
775,383.416305,3.71988,0,5
776,281.399000,3.52830,0,2
777,309.251000,3.87630,1,2


**Combine the 2 data frame**


In [9]:
df_combined = pd.concat([ligands_data, lipinski_df], axis=1)
df_combined

,molecule_chembl_id,canonical_smiles,standard_value,standard_value_M,pIC50,MW,LogP,NumHDonors,NumHAcceptors
0,CHEMBL67010,C/C(=N\C1CCCCC1)NC12CC3CC(CC(C3)C1)C2,72.0,7.200000e-08,7.142668,274.452000,4.29590,1,1
1,CHEMBL542638,C/C(=N\C12CC3CC(CC(C3)C1)C2)Nc1ccccc1C.Cl,6.0,6.000000e-09,8.221849,318.892000,5.21592,1,1
2,CHEMBL544054,C/C(=N\C1CCCCC1)Nc1ccccc1C.Cl,9.0,9.000000e-09,8.045757,266.816000,4.57982,1,1
3,CHEMBL67388,C/C(=N\C12CC3CC(CC(C3)C1)C2)NC12CC3CC(CC(C3)C1)C2,16.0,1.600000e-08,7.795880,326.528000,4.93200,1,1
4,CHEMBL538754,C/C(=N\c1ccccc1C)Nc1ccccc1C.Cl,15.0,1.500000e-08,7.823909,274.795000,4.88724,1,1
...,...,...,...,...,...,...,...,...,...
774,CHEMBL596,CCC(=O)N(c1ccccc1)C1CCN(CCc2ccccc2)CC1,354.0,3.540000e-07,6.450997,336.479000,4.13670,0,2
775,CHEMBL4761695,[2H]C([2H])([2H])Oc1cc(F)cc2c(N3CCN(c4ccccc4F)...,3500.0,3.500000e-06,5.455932,383.416305,3.71988,0,5
776,CHEMBL4297224,CN(C)CC1CCOC1(c1ccccc1)c1ccccc1,860.0,8.600000e-07,6.065502,281.399000,3.52830,0,2
777,CHEMBL4519018,CN1C2CCCC1CC(Nc1ccccc1Br)C2,84.0,8.400000e-08,7.075721,309.251000,3.87630,1,2


In [10]:
# save the data frame to csv.

df_combined.to_csv("../Data/SIR_bioactivity_data_lipinski_04.csv", index=False)

## **Calculating All RDKit Descriptors**

RDKit provides a large collection of molecular descriptors beyond Lipinski's Rule.  
These include physicochemical, topological, electronic, and geometrical features that can be useful for QSAR modeling.

Using `Descriptors.descList`, we can automatically loop through all available descriptors and compute them for a given set of molecules.


In [11]:
from rdkit.Chem import Descriptors

def calculate_all_descriptors(smiles_list):
    """
    Calculate all available RDKit molecular descriptors.

    Parameters
    ----------
    smiles_list : list of str
        List of molecules in SMILES format.

    Returns
    -------
    pd.DataFrame
        DataFrame where each row corresponds to a molecule and
        each column is an RDKit descriptor.
    """
    # List of all descriptor functions in RDKit
    descriptor_names = [desc_name for desc_name, _ in Descriptors.descList]

    results = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            row = {}
            for desc_name, function in Descriptors.descList:
                try:
                    row[desc_name] = function(mol)
                except:
                    row[desc_name] = None
            results.append(row)
        else:
            results.append({desc: None for desc in descriptor_names})

    return pd.DataFrame(results)


In [12]:
# using on my data

ligands_data = pd.read_csv("../Data/SIR_bioactivity_data_pIC50_03.csv")

all_descriptors_df = calculate_all_descriptors(ligands_data["canonical_smiles"])
all_descriptors_df

,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,HeavyAtomMolWt,ExactMolWt,NumValenceElectrons,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,5.017601,5.017601,0.441468,0.441468,0.589148,44.850000,274.452000,244.212,274.240899,112,...,0,0,0,0,0,0,0,0,0,0
1,5.210408,5.210408,0.000000,0.000000,0.586751,36.090909,318.892000,291.676,318.186277,120,...,0,0,0,0,0,0,0,0,0,0
2,4.777908,4.777908,0.000000,0.000000,0.614949,17.111111,266.816000,243.632,266.154976,100,...,0,0,0,0,0,0,0,0,0,0
3,5.450101,5.450101,0.341397,0.341397,0.564390,57.625000,326.528000,292.256,326.272199,132,...,0,0,0,0,0,0,0,0,0,0
4,4.600501,4.600501,0.000000,0.000000,0.612037,10.789474,274.795000,255.643,274.123676,100,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,12.516262,12.516262,0.230496,0.230496,0.791487,15.880000,336.479000,308.255,336.220164,132,...,0,0,0,0,0,0,0,0,0,0
775,14.340202,14.340202,0.155610,-2.775944,0.687352,16.250000,383.416305,362.254,383.163698,142,...,0,0,0,0,0,0,0,0,0,0
776,6.399537,6.399537,0.312708,-0.312708,0.849837,20.809524,281.399000,258.215,281.177964,110,...,0,0,0,0,0,0,0,0,0,0
777,3.724745,3.724745,0.634444,0.634444,0.891786,32.222222,309.251000,288.083,308.088811,98,...,0,0,0,0,0,0,0,0,0,0


In [13]:
all_descriptors_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 779 entries, 0 to 778
Columns: 217 entries, MaxAbsEStateIndex to fr_urea
dtypes: float64(107), int64(110)
memory usage: 1.3 MB


In [14]:
all_descriptors_df.columns

Index(['MaxAbsEStateIndex', 'MaxEStateIndex', 'MinAbsEStateIndex',
       'MinEStateIndex', 'qed', 'SPS', 'MolWt', 'HeavyAtomMolWt', 'ExactMolWt',
       'NumValenceElectrons',
       ...
       'fr_sulfide', 'fr_sulfonamd', 'fr_sulfone', 'fr_term_acetylene',
       'fr_tetrazole', 'fr_thiazole', 'fr_thiocyan', 'fr_thiophene',
       'fr_unbrch_alkane', 'fr_urea'],
      dtype='object', length=217)

**save the data set**


In [15]:
# combine all descriptors with bioactivity data

combine_all_descriptors_df = pd.concat([ligands_data, all_descriptors_df], axis=1)
combine_all_descriptors_df

,molecule_chembl_id,canonical_smiles,standard_value,standard_value_M,pIC50,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,CHEMBL67010,C/C(=N\C1CCCCC1)NC12CC3CC(CC(C3)C1)C2,72.0,7.200000e-08,7.142668,5.017601,5.017601,0.441468,0.441468,0.589148,...,0,0,0,0,0,0,0,0,0,0
1,CHEMBL542638,C/C(=N\C12CC3CC(CC(C3)C1)C2)Nc1ccccc1C.Cl,6.0,6.000000e-09,8.221849,5.210408,5.210408,0.000000,0.000000,0.586751,...,0,0,0,0,0,0,0,0,0,0
2,CHEMBL544054,C/C(=N\C1CCCCC1)Nc1ccccc1C.Cl,9.0,9.000000e-09,8.045757,4.777908,4.777908,0.000000,0.000000,0.614949,...,0,0,0,0,0,0,0,0,0,0
3,CHEMBL67388,C/C(=N\C12CC3CC(CC(C3)C1)C2)NC12CC3CC(CC(C3)C1)C2,16.0,1.600000e-08,7.795880,5.450101,5.450101,0.341397,0.341397,0.564390,...,0,0,0,0,0,0,0,0,0,0
4,CHEMBL538754,C/C(=N\c1ccccc1C)Nc1ccccc1C.Cl,15.0,1.500000e-08,7.823909,4.600501,4.600501,0.000000,0.000000,0.612037,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774,CHEMBL596,CCC(=O)N(c1ccccc1)C1CCN(CCc2ccccc2)CC1,354.0,3.540000e-07,6.450997,12.516262,12.516262,0.230496,0.230496,0.791487,...,0,0,0,0,0,0,0,0,0,0
775,CHEMBL4761695,[2H]C([2H])([2H])Oc1cc(F)cc2c(N3CCN(c4ccccc4F)...,3500.0,3.500000e-06,5.455932,14.340202,14.340202,0.155610,-2.775944,0.687352,...,0,0,0,0,0,0,0,0,0,0
776,CHEMBL4297224,CN(C)CC1CCOC1(c1ccccc1)c1ccccc1,860.0,8.600000e-07,6.065502,6.399537,6.399537,0.312708,-0.312708,0.849837,...,0,0,0,0,0,0,0,0,0,0
777,CHEMBL4519018,CN1C2CCCC1CC(Nc1ccccc1Br)C2,84.0,8.400000e-08,7.075721,3.724745,3.724745,0.634444,0.634444,0.891786,...,0,0,0,0,0,0,0,0,0,0


In [16]:
# save data

all_descriptors_df.to_csv("../Data/SIR_bioactivity_data_rdkit_all_descriptors_05.csv", index=False)

##  **Molecular Fingerprints with RDKit**

After calculating 2D molecular descriptors, the next step is to generate molecular fingerprints, which encode structural information of molecules as fixed-length binary or integer vectors. Fingerprints are widely used in QSAR, similarity searching, and machine learning models for drug discovery.

**Types of Fingerprints**

1. **Morgan Fingerprints (ECFP)**
-  Circular fingerprints that encode atom environments up to a certain radius.

-  Captures local substructures around each atom.

-  Commonly used in QSAR and similarity-based virtual screening (ECFP4 = radius 2).

2. **MACCS Keys**

-  166 pre-defined structural keys representing the presence/absence of specific functional groups.

-  Simple and interpretable fingerprint, often used for ligand-based similarity searching.

3. **Atom Pair Fingerprints**

-  Encodes pairs of atoms and the topological distance between them.

-  Useful for capturing molecular topology information that circular fingerprints may miss.

4. **Topological Torsion Fingerprints**

-  Encodes sequences of four connected atoms along bonds (torsions).

-  Provides information about chain connectivity and local conformations.

**Why Multiple Fingerprints?**

-  Different fingerprints capture different structural features: substructures, functional groups, atom pairs, or torsions.

-  Combining multiple fingerprints improves model robustness and ensures important chemical patterns are represented.

**References**

-  RDKit Documentation: Getting Started in Python

-  Rogers, D., & Hahn, M. (2010). Extended-Connectivity Fingerprints. J. Chem. Inf. Model., 50(5), 742–754. https://doi.org/10.1021/ci100050t

-  Durant, J. L., Leland, B. A., Henry, D. R., & Nourse, J. G. (2002). Reoptimization of MDL Keys for Use in Drug Discovery. J. Chem. Inf. Comput. Sci., 42(6), 1273–1280. https://doi.org/10.1021/ci010132r

In [20]:
# load the data

import pandas as pd

# Load cleaned dataset
data_path = "../Data/SIR_bioactivity_data_pIC50_03.csv"
df = pd.read_csv(data_path)

print( df.shape)
df.head()


(779, 5)


,molecule_chembl_id,canonical_smiles,standard_value,standard_value_M,pIC50
0,CHEMBL67010,C/C(=N\C1CCCCC1)NC12CC3CC(CC(C3)C1)C2,72.0,7.200000e-08,7.142668
1,CHEMBL542638,C/C(=N\C12CC3CC(CC(C3)C1)C2)Nc1ccccc1C.Cl,6.0,6.000000e-09,8.221849
2,CHEMBL544054,C/C(=N\C1CCCCC1)Nc1ccccc1C.Cl,9.0,9.000000e-09,8.045757
3,CHEMBL67388,C/C(=N\C12CC3CC(CC(C3)C1)C2)NC12CC3CC(CC(C3)C1)C2,16.0,1.600000e-08,7.795880
4,CHEMBL538754,C/C(=N\c1ccccc1C)Nc1ccccc1C.Cl,15.0,1.500000e-08,7.823909


##  **Calculate Morgan Fingerprints**

In [48]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
import pandas as pd
import numpy as np

def compute_morgan_fingerprints(smiles_list, radius=2, nBits=1024, verbose=False):
    """
    Compute Morgan (ECFP) fingerprints using RDKit's recommended API.

    Parameters
    ----------
    smiles_list : list of str
        List of molecules in SMILES format.
    radius : int, optional
        The radius of the circular fingerprint (default=2).
    nBits : int, optional
        Length of the fingerprint vector (default=1024).
    verbose : bool, optional
        If True, print progress every 100 molecules.

    Returns
    -------
    pd.DataFrame
        DataFrame with Morgan fingerprints (one row per molecule).
    """
    # Initialize Morgan generator
    morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)

    data = []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            row = np.full(nBits, np.nan)
        else:
            fp = morgan_gen.GetFingerprint(mol)
            arr = np.zeros((nBits,), dtype=int)
            fp.ToBitString()  # Needed internally
            Chem.DataStructs.ConvertToNumpyArray(fp, arr)
            row = arr
        data.append(row)

        if verbose and (i+1) % 100 == 0:
            print(f"{i+1}/{len(smiles_list)} molecules processed")

    columns = [f"Morgan_{i}" for i in range(nBits)]
    return pd.DataFrame(data, columns=columns)


In [50]:
# Load dataset
df = pd.read_csv("../Data/SIR_bioactivity_data_pIC50_03.csv")

# Compute Morgan fingerprints
fps_df = compute_morgan_fingerprints(df["canonical_smiles"], radius=2, nBits=1024, verbose=True)

print("✅ Morgan fingerprints shape:", fps_df.shape)

# Merge with activity values
#df_morgan = pd.concat([df[["smiles", "pIC50"]], fps_df], axis=1)
#df_morgan.to_csv("../Data/sigma1r_morgan.csv", index=False)
#print("✅ Saved Morgan fingerprints dataset")


100/779 molecules processed
200/779 molecules processed
300/779 molecules processed
400/779 molecules processed
500/779 molecules processed
600/779 molecules processed
700/779 molecules processed
✅ Morgan fingerprints shape: (779, 1024)


In [53]:
fps_df.head()


,Morgan_0,Morgan_1,Morgan_2,Morgan_3,Morgan_4,Morgan_5,Morgan_6,Morgan_7,Morgan_8,Morgan_9,...,Morgan_1014,Morgan_1015,Morgan_1016,Morgan_1017,Morgan_1018,Morgan_1019,Morgan_1020,Morgan_1021,Morgan_1022,Morgan_1023
0,0,0,1,0,1,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
2,0,0,1,0,1,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0


##  **Calculate MACCS Fingerprints**

In [55]:
from rdkit import Chem
from rdkit.Chem import MACCSkeys
import pandas as pd

# Example SMILES list
smiles_list = ["CCO", "CCN(CC)CC", "c1ccccc1O"]  # ethanol, triethylamine, phenol
mols = [Chem.MolFromSmiles(s) for s in smiles_list]

# Generate MACCS keys fingerprints and ensure length is 166
fps = [MACCSkeys.GenMACCSKeys(mol) for mol in mols if mol is not None] # Filter out None molecules

# Convert to DataFrame, slicing to ensure 166 columns
maccs_df = pd.DataFrame([list(fp.ToBitString())[:166] for fp in fps], columns=[f"MACCS_{i}" for i in range(166)])


# Add SMILES as reference (optional, and requires handling potential None mols in smiles_list)
# For this example, since we filtered mols, we should filter smiles_list too if adding this column
# maccs_df.insert(0, "SMILES", [smiles_list[i] for i, mol in enumerate([Chem.MolFromSmiles(s) for s in smiles_list]) if mol is not None])

print(maccs_df.head())

  MACCS_0 MACCS_1 MACCS_2 MACCS_3 MACCS_4 MACCS_5 MACCS_6 MACCS_7 MACCS_8  \
0       0       0       0       0       0       0       0       0       0   
1       0       0       0       0       0       0       0       0       0   
2       0       0       0       0       0       0       0       0       0   

  MACCS_9  ... MACCS_156 MACCS_157 MACCS_158 MACCS_159 MACCS_160 MACCS_161  \
0       0  ...         0         1         0         0         1         0   
1       0  ...         0         0         1         0         1         1   
2       0  ...         0         1         0         0         0         0   

  MACCS_162 MACCS_163 MACCS_164 MACCS_165  
0         0         0         1         0  
1         0         0         0         0  
2         1         1         1         1  

[3 rows x 166 columns]


In [39]:
def compute_maccs_fingerprints(smiles_list, verbose=False):
    """
    Calculate MACCS keys fingerprints for a list of molecules.

    Parameters
    ----------
    smiles_list : list of str
        List of molecules in SMILES format.
    verbose : bool
        If True, print progress messages.

    Returns
    -------
    pd.DataFrame
        DataFrame where each row corresponds to a molecule and
        each column is a bit in the MACCS keys fingerprint.
    """
    from rdkit import Chem
    from rdkit.Chem import MACCSkeys
    import numpy as np
    import pandas as pd

    mols = [Chem.MolFromSmiles(smi) for smi in smiles_list]
    data = []
    nBits = 166 # MACCS keys have a fixed length

    for i, mol in enumerate(mols):
        if mol is None:
            row = np.full(nBits, np.nan)
        else:
            try:
                maccs_fp = MACCSkeys.GenMACCSKeys(mol)
                maccs_arr = np.array(maccs_fp)
                if len(maccs_arr) != nBits:
                    print(f"Warning: MACCS keys for molecule {i} has unexpected length {len(maccs_arr)}. Expected {nBits}. Appending NaN row.")
                    # Attempt to handle unexpected length - slicing is common for MACCS
                    if len(maccs_arr) > nBits:
                        maccs_arr = maccs_arr[:nBits]
                    else:
                        # Pad with zeros if less than expected
                        padded_maccs_arr = np.zeros(nBits, dtype=maccs_arr.dtype)
                        padded_maccs_arr[:len(maccs_arr)] = maccs_arr
                        maccs_arr = padded_maccs_arr
                    row = np.full(nBits, np.nan) # Still append NaN row as length was unexpected
                else:
                    row = maccs_arr
            except Exception as e:
                print(f"Error processing molecule {i} for MACCS fingerprint: {e}. Appending NaN row.")
                row = np.full(nBits, np.nan)

        data.append(row)
        if verbose and (i+1) % 100 == 0:
            print(f"{i+1}/{len(mols)} molecules processed for MACCS fingerprints")

    columns = [f"MACCS_{i}" for i in range(nBits)]
    return pd.DataFrame(data, columns=columns)

In [40]:
# Get SMILES column (assuming df is the correct DataFrame)
smiles_list = df["canonical_smiles"].tolist()

# Compute MACCS fingerprints
maccs_fps_df = compute_maccs_fingerprints(smiles_list, verbose=True)

print("✅ MACCS Fingerprints calculated:", maccs_fps_df.shape)
display(maccs_fps_df.head())

100/779 molecules processed for MACCS fingerprints
200/779 molecules processed for MACCS fingerprints
300/779 molecules processed for MACCS fingerprints
400/779 molecules processed for MACCS fingerprints
500/779 molecules processed for MACCS fingerprints
600/779 molecules processed for MACCS fingerprints
700/779 molecules processed for MACCS fingerprints
✅ MACCS Fingerprints calculated: (779, 166)


,MACCS_0,MACCS_1,MACCS_2,MACCS_3,MACCS_4,MACCS_5,MACCS_6,MACCS_7,MACCS_8,MACCS_9,...,MACCS_156,MACCS_157,MACCS_158,MACCS_159,MACCS_160,MACCS_161,MACCS_162,MACCS_163,MACCS_164,MACCS_165
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
def compute_atompair_fingerprints(smiles_list, nBits=1024, verbose=False):
    """
    Calculate Atom Pair fingerprints for a list of molecules.

    Parameters
    ----------
    smiles_list : list of str
        List of molecules in SMILES format.
    nBits : int
        The number of bits in the fingerprint vector.
    verbose : bool
        If True, print progress messages.

    Returns
    -------
    pd.DataFrame
        DataFrame where each row corresponds to a molecule and
        each column is a bit in the Atom Pair fingerprint.
    """
    from rdkit import Chem
    from rdkit.Chem import AllChem
    import numpy as np
    import pandas as pd

    mols = [Chem.MolFromSmiles(smi) for smi in smiles_list]
    data = []

    for i, mol in enumerate(mols):
        if mol is None:
            row = np.full(nBits, np.nan)
        else:
            try:
                atompair_fp = AllChem.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=nBits)
                atompair_arr = np.array(atompair_fp)
                if len(atompair_arr) != nBits:
                    print(f"Warning: Atom Pair fingerprint for molecule {i} has unexpected length {len(atompair_arr)}. Expected {nBits}. Appending NaN row.")
                    row = np.full(nBits, np.nan)
                else:
                    row = atompair_arr
            except Exception as e:
                print(f"Error processing molecule {i} for Atom Pair fingerprint: {e}. Appending NaN row.")
                row = np.full(nBits, np.nan)

        data.append(row)
        if verbose and (i+1) % 100 == 0:
            print(f"{i+1}/{len(mols)} molecules processed for Atom Pair fingerprints")

    columns = [f"AtomPair_{i}" for i in range(nBits)]
    return pd.DataFrame(data, columns=columns)

In [42]:
# Get SMILES column (assuming df is the correct DataFrame)
smiles_list = df["canonical_smiles"].tolist()

# Compute Atom Pair fingerprints
atompair_fps_df = compute_atompair_fingerprints(smiles_list, verbose=True)

print("✅ Atom Pair Fingerprints calculated:", atompair_fps_df.shape)
display(atompair_fps_df.head())

[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION W

100/779 molecules processed for Atom Pair fingerprints
200/779 molecules processed for Atom Pair fingerprints
300/779 molecules processed for Atom Pair fingerprints
400/779 molecules processed for Atom Pair fingerprints


[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION W

500/779 molecules processed for Atom Pair fingerprints
600/779 molecules processed for Atom Pair fingerprints
700/779 molecules processed for Atom Pair fingerprints


[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION WARNING: please use AtomPairGenerator
[08:27:40] DEPRECATION W

✅ Atom Pair Fingerprints calculated: (779, 1024)


,AtomPair_0,AtomPair_1,AtomPair_2,AtomPair_3,AtomPair_4,AtomPair_5,AtomPair_6,AtomPair_7,AtomPair_8,AtomPair_9,...,AtomPair_1014,AtomPair_1015,AtomPair_1016,AtomPair_1017,AtomPair_1018,AtomPair_1019,AtomPair_1020,AtomPair_1021,AtomPair_1022,AtomPair_1023
0,0,0,0,0,1,1,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
1,1,1,0,0,1,1,0,0,1,0,...,0,0,1,1,1,0,1,1,0,0
2,1,1,0,0,1,1,1,0,0,0,...,0,0,1,1,1,0,1,1,0,0
3,0,0,0,0,1,1,1,1,1,0,...,1,0,1,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,1,1,1,0,1,1,0,0


In [43]:
def compute_torsion_fingerprints(smiles_list, nBits=1024, verbose=False):
    """
    Calculate Topological Torsion fingerprints for a list of molecules.

    Parameters
    ----------
    smiles_list : list of str
        List of molecules in SMILES format.
    nBits : int
        The number of bits in the fingerprint vector.
    verbose : bool
        If True, print progress messages.

    Returns
    -------
    pd.DataFrame
        DataFrame where each row corresponds to a molecule and
        each column is a bit in the Topological Torsion fingerprint.
    """
    from rdkit import Chem
    from rdkit.Chem import AllChem
    import numpy as np
    import pandas as pd

    mols = [Chem.MolFromSmiles(smi) for smi in smiles_list]
    data = []

    for i, mol in enumerate(mols):
        if mol is None:
            row = np.full(nBits, np.nan)
        else:
            try:
                torsion_fp = AllChem.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=nBits)
                torsion_arr = np.array(torsion_fp)
                if len(torsion_arr) != nBits:
                    print(f"Warning: Torsion fingerprint for molecule {i} has unexpected length {len(torsion_arr)}. Expected {nBits}. Appending NaN row.")
                    row = np.full(nBits, np.nan)
                else:
                    row = torsion_arr
            except Exception as e:
                print(f"Error processing molecule {i} for Torsion fingerprint: {e}. Appending NaN row.")
                row = np.full(nBits, np.nan)

        data.append(row)
        if verbose and (i+1) % 100 == 0:
            print(f"{i+1}/{len(mols)} molecules processed for Torsion fingerprints")

    columns = [f"Torsion_{i}" for i in range(nBits)]
    return pd.DataFrame(data, columns=columns)

In [44]:
# Get SMILES column (assuming df is the correct DataFrame)
smiles_list = df["canonical_smiles"].tolist()

# Compute Torsion fingerprints
torsion_fps_df = compute_torsion_fingerprints(smiles_list, verbose=True)

print("✅ Torsion Fingerprints calculated:", torsion_fps_df.shape)
display(torsion_fps_df.head())

[08:27:42] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:42] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:42] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:42] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:42] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:42] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:42] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27

100/779 molecules processed for Torsion fingerprints
200/779 molecules processed for Torsion fingerprints
300/779 molecules processed for Torsion fingerprints
400/779 molecules processed for Torsion fingerprints


[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27

500/779 molecules processed for Torsion fingerprints
600/779 molecules processed for Torsion fingerprints
700/779 molecules processed for Torsion fingerprints


[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27:43] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[08:27

✅ Torsion Fingerprints calculated: (779, 1024)


,Torsion_0,Torsion_1,Torsion_2,Torsion_3,Torsion_4,Torsion_5,Torsion_6,Torsion_7,Torsion_8,Torsion_9,...,Torsion_1014,Torsion_1015,Torsion_1016,Torsion_1017,Torsion_1018,Torsion_1019,Torsion_1020,Torsion_1021,Torsion_1022,Torsion_1023
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:
# Concatenate all fingerprint DataFrames
all_fps_df = pd.concat([morgan_fps_df, maccs_fps_df, atompair_fps_df, torsion_fps_df], axis=1)

print("✅ All Fingerprints combined:", all_fps_df.shape)
display(all_fps_df.head())

✅ All Fingerprints combined: (779, 3238)


,Morgan_0,Morgan_1,Morgan_2,Morgan_3,Morgan_4,Morgan_5,Morgan_6,Morgan_7,Morgan_8,Morgan_9,...,Torsion_1014,Torsion_1015,Torsion_1016,Torsion_1017,Torsion_1018,Torsion_1019,Torsion_1020,Torsion_1021,Torsion_1022,Torsion_1023
0,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
